# IBS Wellness Data Exploration

This notebook provides comprehensive exploration and visualization of the IBS wellness dataset.

## Dataset Overview
- **Users**: Patient demographic and profile information
- **Symptom Logs**: Daily symptom tracking data
- **Diet Logs**: Food intake and dietary patterns
- **Food Reactions**: Trigger food identification
- **Medications**: Treatment and medication usage

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Load all datasets
users = pd.read_csv('../data/users.csv')
symptom_logs = pd.read_csv('../data/symptom_logs.csv')
diet_logs = pd.read_csv('../data/diet_logs.csv')
food_reactions = pd.read_csv('../data/food_reactions.csv')
medications = pd.read_csv('../data/medications.csv')
train_data = pd.read_csv('../data/train_data.csv')
val_data = pd.read_csv('../data/val_data.csv')
test_data = pd.read_csv('../data/test_data.csv')

print("Dataset Shapes:")
print(f"Users: {users.shape}")
print(f"Symptom Logs: {symptom_logs.shape}")
print(f"Diet Logs: {diet_logs.shape}")
print(f"Food Reactions: {food_reactions.shape}")
print(f"Medications: {medications.shape}")
print(f"Training Data: {train_data.shape}")
print(f"Validation Data: {val_data.shape}")
print(f"Test Data: {test_data.shape}")

## 1. User Demographics Analysis

In [ ]:
# User demographics overview
print("User Demographics Summary:")
print(users.describe())

# Create demographic visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Age distribution
axes[0, 0].hist(users['age'], bins=20, alpha=0.7, color='skyblue')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')

# Gender distribution
gender_counts = users['gender'].value_counts()
axes[0, 1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%')
axes[0, 1].set_title('Gender Distribution')

# BMI distribution
axes[0, 2].hist(users['bmi'], bins=20, alpha=0.7, color='lightgreen')
axes[0, 2].set_title('BMI Distribution')
axes[0, 2].set_xlabel('BMI')
axes[0, 2].set_ylabel('Frequency')

# IBS type distribution
ibs_counts = users['ibs_type'].value_counts()
axes[1, 0].bar(ibs_counts.index, ibs_counts.values, color='coral')
axes[1, 0].set_title('IBS Type Distribution')
axes[1, 0].set_xlabel('IBS Type')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# Diagnosis years distribution
axes[1, 1].hist(users['diagnosis_years'], bins=15, alpha=0.7, color='gold')
axes[1, 1].set_title('Years Since Diagnosis')
axes[1, 1].set_xlabel('Years')
axes[1, 1].set_ylabel('Frequency')

# Age vs BMI scatter
scatter = axes[1, 2].scatter(users['age'], users['bmi'], c=users['diagnosis_years'], 
                            cmap='viridis', alpha=0.6)
axes[1, 2].set_title('Age vs BMI (colored by diagnosis years)')
axes[1, 2].set_xlabel('Age')
axes[1, 2].set_ylabel('BMI')
plt.colorbar(scatter, ax=axes[1, 2], label='Diagnosis Years')

plt.tight_layout()
plt.show()

## 2. Symptom Patterns Analysis

In [ ]:
# Convert date column to datetime
symptom_logs['date'] = pd.to_datetime(symptom_logs['date'])

# Symptom severity analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Severity score distribution
axes[0, 0].hist(symptom_logs['severity_score'], bins=20, alpha=0.7, color='lightcoral')
axes[0, 0].set_title('Symptom Severity Distribution')
axes[0, 0].set_xlabel('Severity Score (1-10)')
axes[0, 0].set_ylabel('Frequency')

# Stress level distribution
axes[0, 1].hist(symptom_logs['stress_level'], bins=20, alpha=0.7, color='lightblue')
axes[0, 1].set_title('Stress Level Distribution')
axes[0, 1].set_xlabel('Stress Level (1-10)')
axes[0, 1].set_ylabel('Frequency')

# Sleep hours distribution
axes[1, 0].hist(symptom_logs['sleep_hours'], bins=20, alpha=0.7, color='lightgreen')
axes[1, 0].set_title('Sleep Hours Distribution')
axes[1, 0].set_xlabel('Sleep Hours')
axes[1, 0].set_ylabel('Frequency')

# Correlation heatmap
correlation_cols = ['severity_score', 'stress_level', 'sleep_hours']
corr_matrix = symptom_logs[correlation_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Symptom Correlations')

plt.tight_layout()
plt.show()

# Time series analysis
plt.figure(figsize=(15, 8))

# Sample a few users for time series visualization
sample_users = symptom_logs['user_id'].unique()[:5]
colors = plt.cm.Set1(np.linspace(0, 1, len(sample_users)))

for i, user_id in enumerate(sample_users):
    user_data = symptom_logs[symptom_logs['user_id'] == user_id].sort_values('date')
    plt.plot(user_data['date'], user_data['severity_score'], 
             color=colors[i], alpha=0.7, label=f'User {user_id}')

plt.title('Symptom Severity Over Time (Sample Users)')
plt.xlabel('Date')
plt.ylabel('Severity Score')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Dietary Patterns Analysis

In [ ]:
# Diet analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Food type distribution
food_counts = diet_logs['food_type'].value_counts().head(10)
axes[0, 0].barh(food_counts.index, food_counts.values)
axes[0, 0].set_title('Top 10 Food Types')
axes[0, 0].set_xlabel('Frequency')

# Portion size distribution
axes[0, 1].hist(diet_logs['portion_size'], bins=20, alpha=0.7, color='orange')
axes[0, 1].set_title('Portion Size Distribution')
axes[0, 1].set_xlabel('Portion Size')
axes[0, 1].set_ylabel('Frequency')

# Meal type distribution
meal_counts = diet_logs['meal_type'].value_counts()
axes[0, 2].pie(meal_counts.values, labels=meal_counts.index, autopct='%1.1f%%')
axes[0, 2].set_title('Meal Type Distribution')

# Nutritional content analysis
nutrition_cols = ['calories', 'fiber_g', 'sugar_g']
for i, col in enumerate(nutrition_cols):
    axes[1, i].hist(diet_logs[col], bins=20, alpha=0.7)
    axes[1, i].set_title(f'{col.title()} Distribution')
    axes[1, i].set_xlabel(col.title())
    axes[1, i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Food reactions analysis
print("\nFood Reactions Analysis:")
print(f"Total food reactions recorded: {len(food_reactions)}")
print(f"Unique foods causing reactions: {food_reactions['food_name'].nunique()}")

# Top trigger foods
trigger_foods = food_reactions['food_name'].value_counts().head(15)

plt.figure(figsize=(12, 8))
plt.barh(trigger_foods.index, trigger_foods.values, color='salmon')
plt.title('Top 15 Trigger Foods')
plt.xlabel('Number of Reactions')
plt.ylabel('Food Name')
plt.tight_layout()
plt.show()

# Reaction severity distribution
plt.figure(figsize=(10, 6))
severity_counts = food_reactions['reaction_severity'].value_counts()
plt.pie(severity_counts.values, labels=severity_counts.index, autopct='%1.1f%%')
plt.title('Food Reaction Severity Distribution')
plt.show()

## 4. Medication Usage Analysis

In [ ]:
# Medication analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Medication type distribution
med_counts = medications['medication_name'].value_counts().head(10)
axes[0, 0].barh(med_counts.index, med_counts.values, color='lightsteelblue')
axes[0, 0].set_title('Top 10 Medications')
axes[0, 0].set_xlabel('Usage Frequency')

# Dosage distribution
axes[0, 1].hist(medications['dosage_mg'], bins=20, alpha=0.7, color='plum')
axes[0, 1].set_title('Medication Dosage Distribution')
axes[0, 1].set_xlabel('Dosage (mg)')
axes[0, 1].set_ylabel('Frequency')

# Effectiveness rating
effectiveness_counts = medications['effectiveness_rating'].value_counts().sort_index()
axes[1, 0].bar(effectiveness_counts.index, effectiveness_counts.values, color='lightcoral')
axes[1, 0].set_title('Medication Effectiveness Ratings')
axes[1, 0].set_xlabel('Effectiveness Rating (1-10)')
axes[1, 0].set_ylabel('Count')

# Side effects distribution
side_effects_counts = medications['side_effects'].value_counts()
axes[1, 1].pie(side_effects_counts.values, labels=side_effects_counts.index, autopct='%1.1f%%')
axes[1, 1].set_title('Side Effects Distribution')

plt.tight_layout()
plt.show()

print("\nMedication Statistics:")
print(f"Total medication records: {len(medications)}")
print(f"Unique medications: {medications['medication_name'].nunique()}")
print(f"Average effectiveness rating: {medications['effectiveness_rating'].mean():.2f}")
print(f"Percentage with side effects: {(medications['side_effects'] == 'yes').mean()*100:.1f}%")

## 5. Training Data Analysis

In [ ]:
# Training data overview
print("Training Data Analysis:")
print(f"Training set: {len(train_data):,} samples")
print(f"Validation set: {len(val_data):,} samples")
print(f"Test set: {len(test_data):,} samples")
print(f"Unique users in training: {train_data['user_id'].nunique()}")

# Feature distributions in training data
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
numeric_cols = [col for col in numeric_cols if col != 'user_id']

# Create subplots for feature distributions
n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 else axes

for i, col in enumerate(numeric_cols[:len(axes)]):
    if i < len(axes):
        axes[i].hist(train_data[col].dropna(), bins=30, alpha=0.7)
        axes[i].set_title(f'{col} Distribution')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')

# Hide unused subplots
for i in range(len(numeric_cols), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

# Correlation matrix for key features
key_features = ['severity_score', 'stress_level', 'sleep_hours', 'calories', 'fiber_g']
available_features = [col for col in key_features if col in train_data.columns]

if available_features:
    plt.figure(figsize=(10, 8))
    corr_matrix = train_data[available_features].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 6. Data Quality Assessment

In [ ]:
# Data quality analysis
datasets = {
    'Users': users,
    'Symptom Logs': symptom_logs,
    'Diet Logs': diet_logs,
    'Food Reactions': food_reactions,
    'Medications': medications,
    'Training Data': train_data
}

print("Data Quality Assessment:")
print("=" * 60)

quality_summary = []

for name, df in datasets.items():
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    missing_percentage = (missing_cells / total_cells) * 100
    
    quality_summary.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Missing Values': missing_cells,
        'Missing %': missing_percentage
    })
    
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Missing values: {missing_cells:,} ({missing_percentage:.2f}%)")
    
    if missing_cells > 0:
        missing_by_col = df.isnull().sum()
        missing_cols = missing_by_col[missing_by_col > 0]
        print(f"  Columns with missing data: {list(missing_cols.index)}")

# Visualize data quality
quality_df = pd.DataFrame(quality_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Missing data percentage by dataset
axes[0].bar(quality_df['Dataset'], quality_df['Missing %'], color='lightcoral')
axes[0].set_title('Missing Data Percentage by Dataset')
axes[0].set_ylabel('Missing Data %')
axes[0].tick_params(axis='x', rotation=45)

# Dataset sizes
axes[1].bar(quality_df['Dataset'], quality_df['Rows'], color='lightblue')
axes[1].set_title('Dataset Sizes (Number of Rows)')
axes[1].set_ylabel('Number of Rows')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("DATA EXPLORATION SUMMARY")
print("="*60)
print(f"📊 Total users: {users.shape[0]:,}")
print(f"📈 Total symptom logs: {symptom_logs.shape[0]:,}")
print(f"🍽️ Total diet logs: {diet_logs.shape[0]:,}")
print(f"⚠️ Total food reactions: {food_reactions.shape[0]:,}")
print(f"💊 Total medication records: {medications.shape[0]:,}")
print(f"🎯 Training samples: {train_data.shape[0]:,}")
print("="*60)